# Phase 3: Exploratory Data Analysis (EDA) & Manufacturing Analytics

**Project:** AI-Based Predictive Maintenance System for CNC Machines in Automotive Manufacturing Using Industrial Sensor Data  
**Objective:** Perform rigorous, publication-quality exploratory data analysis on the AI4I 2020 dataset to uncover machine operating regimes, failure mechanisms, correlation dynamics, and feature distributions.

---

## 1. Setup, Environment, and Data Loading

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from config import (
    RAW_DATA_PATH, FIGURES_DIR, TABLES_DIR, METRICS_DIR,
    TARGET_COL, FAILURE_TYPE_COLS, CATEGORICAL_COLS, NUMERICAL_COLS, ID_COLS
)
from src.data_loader import load_raw_data
from src.eda import (
    plot_target_distribution,
    plot_machine_type_distribution,
    plot_sensor_distributions,
    plot_sensor_correlation_heatmap,
    plot_failure_mode_distribution,
    plot_failure_wise_boxplots,
    plot_machine_type_vs_failure,
    plot_feature_interactions_vs_failure,
    compute_outlier_summary,
    run_eda_pipeline
)

# Load raw population dataset (10,000 samples)
df = load_raw_data(RAW_DATA_PATH)
print(f"Loaded raw dataset: {df.shape[0]:,} observations, {df.shape[1]} columns.")
df.head()

## 2. Target Distribution: Machine Failure Class Imbalance

Analyzes the prevalence of healthy machine operations ($y=0$) versus failure incidents ($y=1$).

In [ ]:
counts = df[TARGET_COL].value_counts()
percentages = df[TARGET_COL].value_counts(normalize=True) * 100
print(f"Healthy (0): {counts[0]:,} ({percentages[0]:.2f}%)")
print(f"Failure (1): {counts[1]:,} ({percentages[1]:.2f}%)")
print(f"Class Imbalance Ratio: {counts[0]/counts[1]:.2f} : 1")

target_stats = plot_target_distribution(df, FIGURES_DIR / "01_machine_failure_distribution.png")
print(target_stats)

## 3. Machine Product Type Distribution (L, M, H)

Examines the proportion of Low (50%), Medium (30%), and High (20%) quality machine variants in the dataset.

In [ ]:
type_counts = df["Type"].value_counts().reindex(["L", "M", "H"])
type_pct = (df["Type"].value_counts(normalize=True) * 100).reindex(["L", "M", "H"])
print("Machine Type Breakdown:")
for t in ["L", "M", "H"]:
    print(f"Type {t}: {type_counts[t]:,} ({type_pct[t]:.1f}%)")

type_stats = plot_machine_type_distribution(df, FIGURES_DIR / "02_machine_type_distribution.png")

## 4. Continuous Sensor Operating Parameter Distributions

In [ ]:
stats_df = df[NUMERICAL_COLS].describe().T
stats_df["median"] = df[NUMERICAL_COLS].median()
stats_df["skewness"] = df[NUMERICAL_COLS].skew()
stats_df["kurtosis"] = df[NUMERICAL_COLS].kurtosis()
display(stats_df)

plot_sensor_distributions(df, FIGURES_DIR / "05_sensor_distributions.png")

## 5. Sensor Correlation Matrix & Heatmap

Computes pairwise Pearson correlation coefficients ($r$) across sensor variables.

In [ ]:
corr_matrix = df[NUMERICAL_COLS].corr()
display(corr_matrix.round(4))

plot_sensor_correlation_heatmap(df, FIGURES_DIR / "03_sensor_correlation_heatmap.png")

## 6. Failure Mode Breakdown (TWF, HDF, PWF, OSF, RNF)

Evaluates the occurrence of specific failure mechanism flags and verifies their relationship to the primary `Machine failure` indicator.

In [ ]:
modes_counts = {f: df[f].sum() for f in FAILURE_TYPE_COLS}
for f, count in modes_counts.items():
    print(f"{f}: {count} occurrences ({count/len(df)*100:.2f}% of total)")

plot_failure_mode_distribution(df, FIGURES_DIR / "04_failure_mode_distribution.png")

## 7. Comparative Sensor Parameter Analysis (Healthy vs Failed Machines)

In [ ]:
summary_by_fail = df.groupby(TARGET_COL)[NUMERICAL_COLS].agg(["mean", "std", "median"])
display(summary_by_fail)

plot_failure_wise_boxplots(df, FIGURES_DIR / "06_features_by_failure_boxplot.png")

## 8. Machine Product Type vs Observed Failure Rate

In [ ]:
type_fail = df.groupby("Type")[TARGET_COL].agg(
    Total="count",
    Failures="sum",
    Failure_Rate_Pct=lambda x: (x.sum()/len(x))*100
).reindex(["L", "M", "H"])
display(type_fail)

plot_machine_type_vs_failure(df, FIGURES_DIR / "07_machine_type_vs_failure.png")

## 9. Physical Operating Regimes and Failure Clusters

In [ ]:
plot_feature_interactions_vs_failure(df, FIGURES_DIR / "08_feature_interactions_vs_failure.png")

## 10. Sensor Outlier Summary (IQR Method)

In [ ]:
outliers_summary = compute_outlier_summary(df)
display(outliers_summary)

## 11. Run Complete Pipeline and Verify Generated Outputs

In [ ]:
summary = run_eda_pipeline()
print("Pipeline complete. Generated figures:", summary["generated_figures"])
print("Generated tables:", summary["generated_tables"])